<div dir="rtl">
<h1>وقتی گذشته واقعاً ثابت مانده است</h1>
<p>درس 72 از 76 · کدام محاسبهٔ تولید را می‌توان دوباره استفاده کرد؟ · <code dir="ltr">64-cache</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-01/64-cache.html">📖 بازگشت به همین درس</a></p>
<p>خروجی آخرین Query را با K/V نگه‌داشته‌شده بازسازی کنید و مرز اعتبار Cache را ببینید.</p><p>پیش‌نیاز: Q/K/V با شکل (B,H,T,D)، حالت eval و موقعیت‌های ثابت.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر پنجره از چپ بریده و موقعیت‌ها از صفر شماره‌گذاری شوند، آیا K/V قبلی هنوز همان محاسبه را نشان می‌دهند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(12,4,8,2,1,0.0)).eval()
prefix = torch.tensor([[1,2,3]])
extended = torch.tensor([[1,2,3,4]])
with torch.no_grad():
    old_trace,new_trace = {},{}
    model(prefix,trace=old_trace)
    model(extended,trace=new_trace)
old_attention = old_trace['layers'][0]['attention']
new_attention = new_trace['layers'][0]['attention']
torch.testing.assert_close(old_attention['k'],new_attention['k'][:,:,:3])
print('old/new key shapes:',tuple(old_attention['k'].shape),tuple(new_attention['k'].shape))

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>cached_last_attention(q_new,k_old,v_old,k_new,v_new) سه Tensor برگرداند: خروجی Query تازه، K کامل و V کامل. همه شکل (B,H,T,D) دارند و q_new/k_new/v_new فقط یک موقعیت دارند؛ الحاق روی محور زمان و مقیاس 1/sqrt(D) است. آخرین Query اجازهٔ دیدن همهٔ این موقعیت‌ها را دارد.</p>
</div>

In [ ]:
def cached_last_attention(q_new, k_old, v_old, k_new, v_new):
    # TODO: فقط محاسبهٔ آخرین Query
    return None

In [ ]:
def test_exercise():
    args = (new_attention['q'][:,:,-1:],old_attention['k'],old_attention['v'],
            new_attention['k'][:,:,-1:],new_attention['v'][:,:,-1:])
    result = cached_last_attention(*args)
    if result is None:
        return False
    output,keys,values = result
    torch.testing.assert_close(keys,new_attention['k'])
    torch.testing.assert_close(values,new_attention['v'])
    torch.testing.assert_close(output,new_attention['weighted_values'][:,:,-1:])
    query = torch.ones(1,1,1,2)
    empty = torch.empty(1,1,0,2)
    value = torch.tensor([[[[3.0,5.0]]]])
    one = cached_last_attention(query,empty,empty,query,value)
    torch.testing.assert_close(one[0],value)
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: cached_last_attention')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط طول Prefix را از ۱ تا ۳ افزایش دهید؛ K/V ذخیره‌شده را بشمارید. این شمارش، اندازهٔ Cache است نه حافظهٔ کل مدل.</p>
</div>

In [ ]:
with torch.no_grad():
    for length in (1,2,3):
        trace = {}
        model(extended[:,:length],trace=trace)
        attention = trace['layers'][0]['attention']
        print(length,'K+V elements:',attention['k'].numel()+attention['v'].numel())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>حذف قدیمی‌ترین K/V کافی نیست، چون شمارهٔ موقعیت بقیه هم عوض شده است. cache_reusable(old_ids,new_ids,limit) برای فهرست‌های ID و فرض وزن و حالت ثابت، فقط وقتی True بدهد که دقیقاً یک Token به Prefix بدون تغییر اضافه شده و طول از limit نگذشته باشد.</p>
</div>

In [ ]:
with torch.no_grad():
    shifted = {}
    model(torch.tensor([[2,3,4,5]]),trace=shifted)
stale_keys = new_attention['k'][:,:,1:]
recomputed_keys = shifted['layers'][0]['attention']['k'][:,:,:3]
print('stale/recomputed key difference:',(stale_keys-recomputed_keys).abs().max().item())

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def cache_reusable(old_ids, new_ids, limit):
    # TODO: اعتبار Prefix و موقعیت‌ها، پیش از استفادهٔ دوباره
    return None

In [ ]:
def test_repair():
    result = cache_reusable([1,2],[1,2,3],4)
    if result is None:
        return False
    assert result is True
    assert cache_reusable([1,2],[1,4,3],4) is False
    assert cache_reusable([1,2,3,4],[1,2,3,4,5],4) is False
    assert cache_reusable([1,2],[1,2],4) is False
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: cache_reusable')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>Q/K/V از trace واقعی MiniGPT آمده‌اند؛ تابع شما فقط هستهٔ Attention آخرین موقعیت را بازسازی می‌کند، نه Cache کامل همهٔ Layer‌ها. generate پروژه همچنان کل پنجره را دوباره محاسبه می‌کند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>برای تبدیل این تمرین به Cache کامل مدل، کدام وضعیت‌ها را باید برای هر Layer نگه دارید؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-01/64-cache.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/64-cache.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>